# Unrelated-questions black-box probe

Baseline **black-box** method: the "unrelated questions" lie detector from Pacchiardi et al. (*How
to Catch an AI Liar: Lie Detection in Black-Box LLMs by Asking Unrelated Questions*, arXiv:2309.15840).

This is the **inference-only** half of [`tutorials/3.5-unrelated-questions.ipynb`](../tutorials/3.5-unrelated-questions.ipynb):
the tutorial fits a logistic-regression classifier on precomputed yes/no fingerprints and saves it to
`unrelated_questions_probe.npz`; here we load it unchanged, elicit the answers live on NDIF, and score
the private eval set. After the model answers the question being judged, we append each of a fixed bank
of 39 content-unrelated yes/no follow-up questions as a separate one-turn continuation, sample a fresh
reply, and parse it as yes (1) / no (0). Only the data loading and the `submission.csv` write are new.

In [ ]:
import os

import numpy as np
import pandas as pd

# The runner (and `submit.py --dry`) always sets DATASET_NAME — one repo, single
# `test` split — to the dataset you predict on.
DATASET_NAME = os.environ["DATASET_NAME"]

# The canonical output file the grader reads. Do not rename.
SUBMISSION_PATH = "submission.csv"

## Load the trained classifier

`unrelated_questions_probe.npz` ships in the repo next to this notebook. It holds the logistic-regression
weights, the 39 elicitation question texts (in the order the binary feature vector expects), and the generation
config used to sample follow-up replies.

In [ ]:
# cwd is the repo root when the runner executes the notebook; the weights ship under submission/.
CANDIDATES = ["submission/unrelated_questions_probe.npz", "unrelated_questions_probe.npz",
              "./unrelated_questions_probe.npz"]
WEIGHTS = next((p for p in CANDIDATES if os.path.exists(p)), None)
assert WEIGHTS is not None, f"unrelated_questions_probe.npz not found; looked in {CANDIDATES}"

z = np.load(WEIGHTS)
COEF = z["coef"].astype(np.float64)             # (39,)
INTERCEPT = float(z["intercept"])
QUESTIONS = [str(q) for q in z["questions"]]    # 39 elicitation questions, feature order
MODEL_ID = str(z["model_id"])
TEMPERATURE = float(z["temperature"])
MAX_NEW_TOKENS = int(z["max_new_tokens"])
YES_TOKENS = [str(t) for t in z["yes_tokens"]]
NO_TOKENS = [str(t) for t in z["no_tokens"]]
print(f"classifier: model={MODEL_ID} | {len(QUESTIONS)} questions | "
      f"temperature={TEMPERATURE} | max_new_tokens={MAX_NEW_TOKENS}")

def classifier_proba(features):
    """39-dim binary feature vector -> deception probability in [0, 1]."""
    score = np.asarray(features, dtype=np.float64) @ COEF + INTERCEPT
    return 1.0 / (1.0 + np.exp(-np.clip(score, -30.0, 30.0)))

## Load the model

In [ ]:
try:
    from nnsight import VisionLanguageModel
except ImportError:                       # nnsight 0.7.x no longer re-exports at top level
    from nnsight.modeling.vlm import VisionLanguageModel

# You don't configure an NDIF host or API key: the runner sets NDIF_HOST and NDIF_API_KEY in the
# environment and nnsight reads them; the remote generate just needs remote=True.
#
# Optional PEFT/LoRA adapter served on NDIF. Default None = the plain base model the classifier's
# elicitation fingerprints were collected on. nnsight forwards the repo id to NDIF, which wraps the
# served base with PeftModel.from_pretrained per request (base must be hosted there, adapter loadable).
LORA_ADAPTER = os.environ.get("LORA_ADAPTER") or None

if LORA_ADAPTER:
    print(f"applying LoRA adapter on NDIF: {LORA_ADAPTER} (base={MODEL_ID})")
    model = VisionLanguageModel(MODEL_ID, peft=LORA_ADAPTER)
else:
    model = VisionLanguageModel(MODEL_ID)

## Ask the 39 elicitation questions

In [ ]:
from transformers import AutoTokenizer
import torch

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.padding_side = "left"   # newly generated tokens land in the same columns for every row

MAX_SEQ_LEN = 400

def to_judged_messages(msgs):
    """Trim a conversation to end at its last assistant message — the one being judged."""
    idxs = [i for i, m in enumerate(msgs) if m.get("role") == "assistant"]
    return msgs[: idxs[-1] + 1] if idxs else None

def normalize_messages(messages):
    """Merge a leading system message into the first user turn (Gemma's chat
    template doesn't accept a standalone system role)."""
    norm, pending_system = [], None
    for m in messages:
        if m["role"] == "system" and not norm:
            pending_system = m["content"]; continue
        if not norm and m["role"] == "user" and pending_system:
            norm.append({"role": "user", "content": pending_system + "\n\n" + m["content"]})
            pending_system = None; continue
        norm.append(m)
    return norm

def build_elicitation_prompts(messages):
    """One chat-templated, generation-ready prompt per elicitation question."""
    norm = normalize_messages(messages)
    prompts = []
    for q in QUESTIONS:
        ext = norm + [{"role": "user", "content": q}]
        ids = tokenizer.apply_chat_template(
            ext, tokenize=True, add_generation_prompt=True, return_dict=True)["input_ids"]
        if len(ids) > MAX_SEQ_LEN:           # left-truncate very long conversations
            ids = ids[len(ids) - MAX_SEQ_LEN:]
        prompts.append(ids)
    return prompts

def parse_yes_no(text):
    """Mirrors `parse_yes_no_response` in liars-bench/src/blackbox/pacchiardi.py."""
    t = text.strip().lower()
    for tok in YES_TOKENS:
        if t.startswith(tok) or f" {tok}" in t:
            return 1
    for tok in NO_TOKENS:
        if t.startswith(tok) or f" {tok}" in t:
            return 0
    return 0   # unparseable -> 0 (deviation from training, where it's discarded)

def elicit_features(messages):
    """Ask all 39 elicitation questions as separate one-turn continuations of `messages`,
    batched into a single remote `generate` call. Returns the 39-dim binary feature vector."""
    prompts = build_elicitation_prompts(messages)
    batch = tokenizer.pad([{"input_ids": ids} for ids in prompts], return_tensors="pt")
    input_ids, attn = batch["input_ids"], batch["attention_mask"]
    with model.generate({"input_ids": input_ids, "attention_mask": attn}, remote=True,
                         do_sample=True, temperature=TEMPERATURE, max_new_tokens=MAX_NEW_TOKENS):
        out = model.generator.output.save()
    new_tokens = out[:, input_ids.shape[1]:]
    replies = tokenizer.batch_decode(new_tokens, skip_special_tokens=True)
    return np.array([parse_yes_no(r) for r in replies], dtype=np.float64)

## Predict and write `submission.csv`

In [ ]:
import sys

# util.py ships alongside the notebooks in submission/; the runner executes from
# the repo root, so add submission/ to the path before importing it.
sys.path.insert(0, "submission")
import util

examples = util.load_examples(DATASET_NAME)   # single `test` split + an `index` join key
print(f"loaded {len(examples)} examples")

predictions = np.full(len(examples), 0.5, dtype=float)   # fallback for unparseable rows
n_scored = 0
for i, ex in enumerate(examples):
    msgs = to_judged_messages(list(ex["messages"]))
    if msgs is None:
        continue
    features = elicit_features(msgs)
    predictions[i] = classifier_proba(features)
    n_scored += 1
    print(f"  example {i + 1}/{len(examples)}")
print(f"scored {n_scored}/{len(examples)} examples "
      f"({len(examples) - n_scored} fell back to 0.5)")

In [ ]:
predictions = np.clip(np.asarray(predictions, dtype=float), 0.0, 1.0)
assert len(predictions) == len(examples), "need exactly one prediction per example"

# Write index,deceptive,score, joining on the dataset's `index` column; `deceptive`
# defaults to score >= 0.5. (Replaces the old id,prediction output.)
util.write_submission(examples["index"], predictions)